# Example 8 — PEPSI legacy line-window validation

## What this example teaches

This example preserves the PEPSI legacy line-window validation workflow in the numbered path. It reads bundled PEPSI `.nor` spectra, turns the historical line windows into a `SpectrumCollection`, and shows the command that runs the maintained legacy PHOENIX fit.

## Requirements

The inspection cells use only bundled PEPSI spectra. The optional full legacy fit requires a configured local PHOENIX library and is run through `scripts/pepsi_fit_smoketest.py` so the optimizer is maintained in one place.

## Expected outputs

You should see the PEPSI reader assumptions, a prepared window summary, and an observed-window plot. This is a compatibility/regression workflow, not a generic final-analysis recipe for every PEPSI product.


## 0. Imports and controls

The PEPSI reader name describes the data product. The `.dxt.nor` suffix alone is not enough to infer wavelength frame for every PEPSI release, so the working wavelength hypothesis is kept explicit.


In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import Spyctres as sp

ROOT = Path.cwd().resolve()
pepsi_paths = [
    sp.example_data_path("pepsir.20230603.009.dxt.nor"),
    sp.example_data_path("pepsir.20230603.010.dxt.nor"),
]

reader = "pepsi_nor"
wave_hypothesis = "air"
legacy_halfwidth_A = 10.0
window_pad_A = 2.0
RUN_LEGACY_FIT = False


## 1. Read the PEPSI spectra

`sp.read_spectrum(..., reader="pepsi_nor")` returns the same canonical spectrum container used by the rest of Spyctres. The reader records metadata such as wavelength medium, observer frame, stellar-rest status, and PEPSI velocity keywords when present. It does not silently apply `SSBVEL` or infer a stellar-rest correction from the filename.


In [ ]:
print("Available readers include:", ", ".join(sp.list_readers()))
print(sp.get_reader_info(reader).to_metadata())

raw_segments = [sp.read_spectrum(path, reader=reader) for path in pepsi_paths]
for segment in raw_segments:
    print("
", segment.name)
    print(segment.summary())


## 2. Build the legacy line windows

The historical validation compared a small set of PEPSI line windows rather than fitting the whole red spectrum. Spyctres keeps that operation in `sp.recipes.build_pepsi_legacy_segments()` so the notebook does not need to define its own window-building logic.


In [ ]:
input_segments, fit_segments, window_defs_air = sp.recipes.build_pepsi_legacy_segments(
    raw_segments,
    wave_hypothesis=wave_hypothesis,
    halfwidth_A=legacy_halfwidth_A,
    window_pad_A=window_pad_A,
)

collection = sp.SpectrumCollection(
    fit_segments,
    name="example8_pepsi_legacy_line_windows",
    meta={
        "workflow": "example8_pepsi_legacy_linefit_validation",
        "wave_hypothesis": wave_hypothesis,
        "legacy_window_defs_air": [list(item) for item in window_defs_air],
    },
)

summary = collection.summary()
print("Prepared windows:", summary["n_segments"])
print("Total pixels:", summary["n_pixels"])
print("Valid fraction:", f"{summary['valid_fraction']:.3f}")
for item in summary["segments"]:
    print(
        f"{item['name']:<18} {item['n_valid_pixels']:>4}/{item['n_pixels']:<4} "
        f"{item['wavelength_range_A'][0]:.1f}-{item['wavelength_range_A'][1]:.1f} Å"
    )


## 3. Inspect the prepared windows before fitting

Gray points are not currently used by the legacy comparison. This visual check is deliberately separate from the fit: if a PEPSI product has different wavelength or frame conventions, inspect the headers/release notes before changing `wave_hypothesis` or applying velocity corrections.


In [ ]:
nseg = len(collection.segments)
fig, axes = plt.subplots(
    nseg,
    1,
    figsize=(11.5, max(2.0, 1.55 * nseg)),
    sharex=False,
    constrained_layout=True,
)
axes = np.atleast_1d(axes)
for ax, segment in zip(axes, collection.segments):
    wave = np.asarray(segment.wave, dtype=float)
    flux = np.asarray(segment.flux, dtype=float)
    valid = np.asarray(segment.valid_mask, dtype=bool) & np.isfinite(wave) & np.isfinite(flux)
    ax.plot(wave[valid], flux[valid], color="0.15", lw=0.8, label="used")
    if np.any(~valid):
        ax.plot(wave[~valid], flux[~valid], ".", color="0.70", ms=2, label="not used")
    ax.set_ylabel(str(segment.name))
    ax.grid(alpha=0.18)
axes[-1].set_xlabel("Wavelength [Å]")
axes[0].set_title("PEPSI legacy line windows; no PHOENIX fit has been run")
axes[0].legend(frameon=False, fontsize=8, loc="best")
plt.show()


## 4. Optional: run the maintained legacy fit

The full validation fit is intentionally delegated to `scripts/pepsi_fit_smoketest.py`. That script uses the shared PEPSI recipe and PHOENIX backend, prints progress, and is the regression target we keep synchronized with the package.


In [ ]:
command = [
    sys.executable,
    str(ROOT / "scripts" / "pepsi_fit_smoketest.py"),
    "--preset",
    "pepsi_legacy_red_fast",
    str(pepsi_paths[0]),
    str(pepsi_paths[1]),
]

if RUN_LEGACY_FIT:
    subprocess.run(command, check=True)
else:
    print("Legacy PHOENIX fit is disabled. To run it later:")
    print(" ".join(command))


## Interpretation checklist

- PEPSI `.nor` products from different releases can have different wavelength conventions.
- For PETS/NASA-style stellar-rest products, do not apply `SSBVEL`/`OBSVEL` again.
- For generic local PEPSI products, verify FITS headers or release documentation before applying velocity corrections.
- This example validates a legacy line-window path; use the reviewed-analysis examples for ordinary PHOENIX interpretation discipline.
